# Lesson 01 - 介紹 AI 代理人

歡迎來到 **AI 代理人入門** 課程的第一課！

**AI 代理人** 是一個使用大型語言模型（LLM）作為其推理引擎，並能在現實世界中採取<em>行動</em>的程式——調用 API、查詢資料庫或運行代碼——以代表使用者完成目標。

在這個筆記本中，你將構建第一個代理人：一個推薦度假目的地的<strong>旅行代理人</strong>。同時你將學習如何：

1. 使用 **Microsoft Agent Framework** 連接到 Microsoft Foundry 代理服務。
2. 給代理人一個<strong>工具</strong>——一個它可以調用的普通 Python 函數。
3. 運行代理人並檢查其回應。
4. 逐個 Token 流式傳輸代理人的回應。


## 設定

在執行此筆記本之前，請確保您已經：

1. **擁有一個 Microsoft Foundry 專案** 並已部署聊天模型（例如 `gpt-5-mini`）。
2. **已使用 Azure CLI 登入** — 在終端機中執行 `az login`。
3. **設定所需的環境變數：**
   - `AZURE_AI_PROJECT_ENDPOINT` — 您的 Microsoft Foundry 專案端點。
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — 您部署的模型名稱。

以下程式碼區塊會安裝您所需的 Python 套件。


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## 創建你的第一個代理人

一個代理人需要兩樣東西：

- <strong>指示</strong>，告訴它<em>它是誰</em>以及<em>如何行為</em>（系統提示）。
- <strong>工具</strong> — 用 `@tool` 裝飾的 Python 函數，代理人可以調用這些函數來獲取信息或執行操作。

以下我們定義了一個簡單的工具，可以返回熱門旅遊目的地的列表。當用戶請求旅遊建議時，代理人將使用這個工具。


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## 流式回應

為了提供更互動的體驗，你可以<strong>串流</strong>代理的回應。代理會隨著文字區塊生成即時傳出，而不是等待完整回覆。這在聊天介面中特別有用，因為你可以即時顯示輸出結果。


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## 摘要

在這課中你學會了如何：

- <strong>建立一個提供者</strong>，透過 `FoundryChatClient` 連接到 Microsoft Foundry Agent Service。
- **使用 `@tool` 裝飾器定義工具**，讓代理能呼叫你的 Python 函數。
- <strong>運行代理</strong>，使用使用者訊息並打印其回應。
- <strong>串流回應</strong>，以實現即時輸出。

在下一課我們將更深入探討代理框架，並學習如何給代理更強大的工具和多步推理能力。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們力求準確，但請注意，自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議尋求專業人工翻譯。我們不對因使用本翻譯而引起的任何誤解或曲解承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
